# Phase 6: Deployment

## Step 1: Batch Prediction Script
This section simulates how the company would use our model on a monthly basis to find at-risk customers.

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib

print("Loading RAW dataset for training...")

df_raw = pd.read_csv('../data/raw/Dataset.csv')
y_df = pd.read_pickle('../data/processed/target.pkl')
y = y_df.values.ravel()

leakage_cols = ['Customer ID', 'Churn Category', 'Churn Label', 'Churn Value', 'Churn Reason', 'Churn']
X = df_raw.drop(columns=[col for col in leakage_cols if col in df_raw.columns])

# 4. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Automatically identify column types in the RAW data
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Fixes missing numbers
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Fixes missing text
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # Handles unknown/messy text
])

# 7. Combine the cleaners
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# 8. Assemble the final pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
])

print("Training the full pipeline (this may take a moment)...")
full_pipeline.fit(X_train, y_train)

# 10. Save the smart pipeline
joblib.dump(full_pipeline, 'churn_model_production.pkl')
print("Full pipeline trained and saved successfully as churn_model_production.pkl")

Loading RAW dataset for training...
Training the full pipeline (this may take a moment)...
Full pipeline trained and saved successfully as churn_model_production.pkl


In [4]:
import pandas as pd
import os
import joblib

model_path = 'churn_model_production.pkl'
raw_data_path = '../data/raw/Dataset.csv'
output_csv = '../data/processed/monthly_retention_leads.csv'

print(f"Loading pipeline from '{model_path}'...")
pipeline = joblib.load(model_path)

print("Loading raw dataset for batch prediction...")
raw_df = pd.read_csv(raw_data_path)

print("Simulating Batch Prediction on new data...\n")

leakage_cols = [ 'Churn Category', 'Churn Label', 'Churn Value', 'Churn Reason', 'Churn']
X_new = raw_df.drop(columns=[col for col in leakage_cols if col in raw_df.columns])

churn_probs = pipeline.predict_proba(X_new)[:, 1]

deployment_df = raw_df.copy()
deployment_df['Churn_Prob'] = churn_probs

high_risk = deployment_df[deployment_df['Churn_Prob'] > 0.75].copy()

if 'Monthly Charge' in high_risk.columns:
    high_risk = high_risk.rename(columns={'Monthly Charge': 'Monthly_Charge'})

high_risk['Discount_Rate'] = high_risk['Churn_Prob'].apply(
    lambda prob: 0.05 if prob <= 0.85 else (0.10 if prob <= 0.95 else 0.15)
)

high_risk['Monthly_Discount'] = high_risk['Monthly_Charge'] * high_risk['Discount_Rate']
high_risk['Annual_Offer_Cost'] = high_risk['Monthly_Discount'] * 12

if 'CLTV' not in high_risk.columns:
    print("Warning: 'CLTV' column missing from raw data. Defaulting to 0.")
    high_risk['CLTV'] = 0

high_risk['Expected_Loss'] = high_risk['Churn_Prob'] * high_risk['CLTV']
high_risk['Net_ROI'] = high_risk['Expected_Loss'] - high_risk['Annual_Offer_Cost']

high_risk = high_risk.sort_values('Expected_Loss', ascending=False)
high_risk['Priority_Rank'] = range(1, len(high_risk) + 1)

output_cols = [
    'Priority_Rank', 'Churn_Prob', 'Contract',
    'CLTV', 'Monthly_Charge', 'Discount_Rate',
    'Monthly_Discount', 'Annual_Offer_Cost',
    'Expected_Loss', 'Net_ROI'
]

missing_cols = [col for col in output_cols if col not in high_risk.columns]
if missing_cols:
    print(f"Warning: Missing requested columns skipped: {missing_cols}")
safe_output_cols = [col for col in output_cols if col in high_risk.columns]

os.makedirs(os.path.dirname(output_csv), exist_ok=True)
high_risk[safe_output_cols].to_csv(output_csv, index=False)

print(f"Customers flagged (>75% risk) : {len(high_risk)}")
print(f"Total revenue at risk: ${high_risk['Expected_Loss'].sum():,.0f}")
print(f"Total cost of all offers: ${high_risk['Annual_Offer_Cost'].sum():,.0f}")
print(f"Net ROI if all retained: ${high_risk['Net_ROI'].sum():,.0f}")
print("\nTop 5 Priority Customers:")
print(high_risk[safe_output_cols].head().to_string(index=False))

Loading pipeline from 'churn_model_production.pkl'...
Loading raw dataset for batch prediction...
Simulating Batch Prediction on new data...

Customers flagged (>75% risk) : 1391
Total revenue at risk: $5,016,241
Total cost of all offers: $116,913
Net ROI if all retained: $4,899,327

Top 5 Priority Customers:
 Priority_Rank  Churn_Prob       Contract  CLTV  Monthly_Charge  Discount_Rate  Monthly_Discount  Annual_Offer_Cost  Expected_Loss   Net_ROI
             1        0.98 Month-to-Month  5948          78.520           0.15           11.7780           141.3360        5829.04 5687.7040
             2        0.97 Month-to-Month  5980          77.688           0.15           11.6532           139.8384        5800.60 5660.7616
             3        0.92 Month-to-Month  6259          99.060           0.10            9.9060           118.8720        5758.28 5639.4080
             4        0.97 Month-to-Month  5913          99.996           0.15           14.9994           179.9928        57

## Step 2: Deployment Strategy

- **Delivery Method**: A Batch Script. Every month, the database will feed new customer data into this Python script, which generates a CSV of "leads."
- **Operational Integration**: This CSV is sent automatically to the Customer Success Team and the Marketing Department.
- **Automation**: We suggest using a tool like GitHub Actions or a Cron Job to run this prediction every 30 days.

## Step 3: Monitoring & Maintenance

- **Model Drift**: We will check the model's accuracy every 6 months. If customer behavior changes (e.g., a new competitor enters the market), we will retrain the model with fresh data.
- **Feedback Loop**: The "Retention Team" will record if the 5% discount actually worked. This data will be used to improve the next version of the model.

## Step 4: Final Summary for CW (The Poster/Report)

- **The Threshold**: "We deploy a 75% probability threshold to prioritize the most urgent cases."
- **The Value**: "We target 199 high-risk customers identified in our test results. By focusing on these specific   individuals, we ensure the retention budget is spent with 92.51% confidence, drastically reducing 'false alarms."
- **The Impact**: "By automating the tiered discount offer, we project a 
  Net ROI of ~$691,057 by preventing churn before it happens."

---

## Key Takeaway on Iteration

While the six CRISP-DM phases are presented **sequentially** in these notebooks, in practice the process is **highly iterative**. Real-world data science projects rarely follow a straight line from Phase 1 to Phase 6.

Common iteration patterns include:

- **Modelling → Data Preparation:** If the model performs poorly, you may return to Phase 3 to engineer new features, clean data differently, or acquire more data.
- **Evaluation → Modelling:** If the model does not meet the business success criteria, you may go back to Phase 4 to try different algorithms or tune hyperparameters.
- **Deployment → Evaluation:** If the deployed model underperforms in production (data drift), you may return to Phase 5 to re-evaluate and then to Phase 3/4 to retrain.
- **Any Phase → Business Understanding:** New findings in later phases may redefine the business objectives or success criteria originally set in Phase 1.

```
┌───────────────────────────────────────────────────────────┐
│                     CRISP-DM Lifecycle                    │
│                                                           │
│   Phase 1 ──► Phase 2 ──► Phase 3 ──► Phase 4            │
│   Business    Data        Data        Modelling           │
│   Under.      Under.      Prep.           │               │
│     ▲                       ▲              │               │
│     │                       │              ▼               │
│     │                       └──────── Phase 5             │
│     │                                Evaluation           │
│     │                                    │                │
│     │                                    ▼                │
│     └──────────────────────────────  Phase 6              │
│                                     Deployment            │
└───────────────────────────────────────────────────────────┘
```

> **Remember:** Iteration is not failure — it is the *expected* workflow. Each cycle through the process deepens your understanding of both the data and the business problem, ultimately leading to a better solution.